# Notebook 35 - Trigger isolation for the BatchNorm collapse (CICIoT2023)

The collapse occurs on CICIoT2023 in pruned and unpruned models and never on CIC-IoMT-2024 under the same recipe. This notebook tests what causes it, on the cells where it happens, one factor at a time, with criteria fixed before any result (stage 2).

**T1, input clipping.** The corpus that never collapses is percentile-clipped at 0.1 and 99.9 before standardisation. Stage 3 first measures how extreme the standardised CICIoT2023 features are; stage 5 then clips them the same way at recovery and evaluation, on all six collapsing cells.

**T2, no running statistics.** Dense models rebuilt with GroupNorm in place of BatchNorm, trained under the NB29 recipe, recovered on the collapsing seeds. With no running statistics there is no dual mode; the signal is whether eval-mode benign escalation ever spikes. If it never does, the weights never went bad and the stored statistics were the whole mechanism.

**T3, subset versus order.** The seed fixes both which rows are in the recovery subset and how they are batched; this separates the two.

**T4, unweighted loss.** The one recipe factor never ablated.

**T5, per-batch trace.** On the baseline shallow Fisher s307 run: each batch's maximum |z| and the running variance after the step, with benign escalation sampled every 25 batches, so a variance jump can be matched to the batch that caused it.

**Stages.** 1 bootstrap, 2 pre-registration, 3 data, teachers, helpers, clip bounds and the extreme-value diagnostic, 4 baseline reproduction with the trace (stops if no collapse reproduces), 5 T1, 6 T2 (trains two GroupNorm teachers, cached under `models/ciciot2023/`), 7 T3, 8 T4, 9 verdict, 10 figures. Every training stage resumes per completed cell. No test access. GPU required.

In [ ]:
# Stage 1 - bootstrap
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, sys, json, copy, hashlib
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, yaml
import matplotlib.pyplot as plt

REPO = Path("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from src.saber.bridge_ciciot import load_bridge
from src.saber.taxonomy import ciciot2023_taxonomy, DEFAULT_COST_PROFILES
from src.saber.metrics import full_model_audit, action_weighted_boundary_inversion_rate
from src.saber.surgery import prune_cnn1d_channels

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
R = REPO / "results/saber"
OUT = R / "35_trigger_isolation"; OUT.mkdir(parents=True, exist_ok=True)
MODEL_DIR = REPO / "models/ciciot2023"
print("device:", DEVICE)


In [ ]:
# Stage 2 - pre-registration
PREREG = {
    "arm": "T_trigger_isolation_ciciot2023",
    "cells": ("the four pruned cells that collapse in NB27/28/30 (shallow fisher s307, shallow saber_v2 s307, deep fisher s401, "
              "deep saber_v2 s401) and the two unpruned cells that collapse in NB31 (dense shallow s307, dense deep s401); "
              "recovery: 8 (shallow) / 6 (deep) units on the seeded 10% subset, Adam 1e-3, batch 1024, inverse-sqrt class weighting"),
    "collapse_definition": "as in NB30/31: eval-mode benign escalation > 0.10 with batch-statistics escalation <= 0.10",
    "conditions": {
        "T1_input_clipping": ("features clipped to their 0.1 / 99.9 training percentiles before the model, the preprocessing of the "
                              "corpus that never collapsed; applied at recovery and evaluation; all six cells"),
        "T2_no_running_statistics": ("dense models rebuilt with GroupNorm(1, C) in place of BatchNorm, trained under the NB29 recipe, "
                                     "recovered under the identical seeded subsets and orders (shallow s307, deep s401); with no "
                                     "running statistics there is no dual mode, so the signal is an eval-mode benign escalation "
                                     "spike > 0.10 at any epoch"),
        "T3_subset_vs_order": ("shallow fisher: (a) subset fixed at seed 307, batch order varied over three seeds; (b) batch-order "
                               "seed fixed at 307, subset varied over three seeds"),
        "T4_unweighted_loss": "plain cross-entropy on the four pruned cells",
        "T5_per_batch_trace": ("baseline shallow fisher s307: per batch, the batch's maximum |z| and the maximum BatchNorm running "
                               "variance after the step; eval-mode benign escalation every 25 batches")},
    "claims": {
        "T1": "input clipping removes every normalisation collapse in all six cells",
        "T2": "GroupNorm dense models show no eval-mode benign escalation spike > 0.10 at any epoch for seeds 307 and 401",
        "T3": ("collapses track the subset: they occur in at least two of three order variants with the subset fixed and in at most "
               "one of three subset variants with the order fixed; the reverse pattern is reported as tracking the order; neither is "
               "reported as such"),
        "T4": "removing class weighting does not remove the collapse in the four pruned cells (at least one collapse remains)",
        "T5": ("descriptive: the three largest single-batch jumps in maximum running variance coincide with the three batches of "
               "highest maximum |z| in the same unit")},
    "reporting_rule": "each claim reported as held or not held",
    "no_test_access": True, "no_new_selection": True,
}
(OUT / "T_PREREGISTRATION.json").write_text(json.dumps(PREREG, indent=2))
print(json.dumps(PREREG, indent=2))
COLLAPSE_B2A = 0.10; SUBSET_FRACTION = 0.10
E_MAX = {"shallow": 8, "deep": 6}
PRUNED_CELLS = [("shallow", "fisher", 307), ("shallow", "saber_v2", 307), ("deep", "fisher", 401), ("deep", "saber_v2", 401)]
DENSE_CELLS = [("shallow", 307), ("deep", 401)]


In [ ]:
# Stage 3 - data, teachers, helpers, clip bounds and the extreme-value diagnostic
TRAIN_LOADER, VAL_LOADER, _T, SHALLOW_TEACHER, CLASS_NAMES = load_bridge()
taxonomy = ciciot2023_taxonomy(CLASS_NAMES)
robust_graph = pd.read_csv(R / "14_risk_graph/asvg_edges_robust.csv")
N_CLASSES = len(CLASS_NAMES)
SABER_CFG = yaml.safe_load(open(REPO / "config/saber.yaml"))
MIN_W = {"shallow": int(SABER_CFG["groups"]["minimum_remaining_per_layer"]), "deep": 8}


def make_cnn(norm, n_classes=34):
    def N(c):
        return nn.BatchNorm1d(c) if norm == "bn" else nn.GroupNorm(1, c)
    class CNN1D(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv = nn.Sequential(nn.Conv1d(1, 64, 3, padding=1), nn.ReLU(), N(64), nn.Conv1d(64, 128, 3, padding=1), nn.ReLU(), N(128))
            self.pool = nn.AdaptiveAvgPool1d(1); self.head = nn.Linear(128, n_classes)
        def forward(self, x):
            if x.dim() == 2:
                x = x.unsqueeze(1)
            return self.head(self.pool(self.conv(x.float())).squeeze(-1))
    class DeepCNN1D(nn.Module):
        def __init__(self):
            super().__init__()
            def blk(i, o):
                return [nn.Conv1d(i, o, 3, padding=1), nn.ReLU(), N(o)]
            self.conv = nn.Sequential(*blk(1, 64), *blk(64, 128), nn.MaxPool1d(2), *blk(128, 128), *blk(128, 256))
            self.pool = nn.AdaptiveAvgPool1d(1); self.head = nn.Linear(256, n_classes)
        def forward(self, x):
            if x.dim() == 2:
                x = x.unsqueeze(1)
            return self.head(self.pool(self.conv(x.float())).squeeze(-1))
    return {"shallow": CNN1D, "deep": DeepCNN1D}


TEACHERS = {"shallow": SHALLOW_TEACHER.to(DEVICE).eval()}
_dt = make_cnn("bn")["deep"]()
_dt.load_state_dict(torch.load(MODEL_DIR / "deepcnn1d_g5_seed0.pt", map_location="cpu", weights_only=False)["state_dict"])
TEACHERS["deep"] = _dt.to(DEVICE).eval()

Xv, Yv = VAL_LOADER.dataset.tensors; VAL_Y_ALL = Yv.numpy()
_rng = np.random.default_rng(0)
_idx = np.concatenate([_rng.permutation(np.where(VAL_Y_ALL == c)[0])[:4000] for c in range(N_CLASSES) if (VAL_Y_ALL == c).sum() > 0])
_idx = np.random.default_rng(12345).permutation(_idx)
EX_X = Xv[_idx].to(DEVICE); EX_Y = VAL_Y_ALL[_idx]; EXAMPLE_INPUT = Xv[:8].float().to(DEVICE)
Xt, Yt = TRAIN_LOADER.dataset.tensors; N_TRAIN = len(Xt)
_counts = np.bincount(Yt.numpy(), minlength=N_CLASSES); _w = np.zeros_like(_counts, dtype=np.float64)
_w[_counts > 0] = 1.0 / np.sqrt(_counts[_counts > 0]); _w[_counts > 0] /= _w[_counts > 0].mean()
CLASS_W = torch.tensor(_w, dtype=torch.float32, device=DEVICE)

# --- extreme-value diagnostic on the standardised CICIoT2023 features ---
_sub = Xt[torch.randperm(N_TRAIN, generator=torch.Generator().manual_seed(2026))[:400_000]].numpy()
LO = torch.tensor(np.percentile(_sub, 0.1, axis=0), dtype=torch.float32, device=DEVICE)
HI = torch.tensor(np.percentile(_sub, 99.9, axis=0), dtype=torch.float32, device=DEVICE)
absmax_train = Xt.abs().max(dim=0).values.numpy()
frac_gt10 = float((Xt.abs() > 10).any(dim=1).float().mean()); frac_gt100 = float((Xt.abs() > 100).any(dim=1).float().mean())
diag = {"n_features": int(Xt.shape[1]), "max_abs_z_per_feature": absmax_train.round(2).tolist(),
        "features_with_max_abs_z_over_100": int((absmax_train > 100).sum()), "fraction_rows_any_abs_z_over_10": frac_gt10,
        "fraction_rows_any_abs_z_over_100": frac_gt100, "clip_lo": LO.cpu().numpy().round(4).tolist(), "clip_hi": HI.cpu().numpy().round(4).tolist()}
for seed in (307, 401, 101):
    sub = torch.randperm(N_TRAIN, generator=torch.Generator().manual_seed(seed))[: int(N_TRAIN * SUBSET_FRACTION)]
    xs = Xt[sub]; diag[f"subset_s{seed}_max_abs_z"] = float(xs.abs().max()); diag[f"subset_s{seed}_rows_abs_z_over_100"] = int((xs.abs() > 100).any(dim=1).sum())
json.dump(diag, open(OUT / "extreme_value_diagnostic.json", "w"), indent=2)
print("max |z| over all features:", float(absmax_train.max()), "| features with max |z| > 100:", diag["features_with_max_abs_z_over_100"],
      "| rows with any |z| > 100:", f"{frac_gt100:.2e}")
for seed in (307, 401, 101):
    print(f"  subset seed {seed}: max |z| {diag[f'subset_s{seed}_max_abs_z']:.1f}, rows with |z| > 100: {diag[f'subset_s{seed}_rows_abs_z_over_100']}")


class Clipped(nn.Module):
    def __init__(self, inner, lo, hi):
        super().__init__(); self.inner = inner; self.register_buffer("lo", lo.clone()); self.register_buffer("hi", hi.clone())
    def forward(self, x):
        return self.inner(torch.maximum(torch.minimum(x, self.hi), self.lo))


def forward_logits(model):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(EX_X[i:i + 8192]).cpu() for i in range(0, len(EX_X), 8192)]).numpy()


def forward_logits_batch_stats(model):
    saved = {n: (m.running_mean.clone(), m.running_var.clone(), m.momentum, m.num_batches_tracked.clone())
             for n, m in model.named_modules() if isinstance(m, nn.BatchNorm1d) and m.running_mean is not None}
    model.train()
    for n, m in model.named_modules():
        if n in saved: m.momentum = 0.0
    with torch.no_grad():
        out = torch.cat([model(EX_X[i:i + 8192]).cpu() for i in range(0, len(EX_X), 8192)]).numpy()
    for n, m in model.named_modules():
        if n in saved:
            rm, rv, mom, nbt = saved[n]; m.running_mean.copy_(rm); m.running_var.copy_(rv); m.momentum = mom; m.num_batches_tracked.copy_(nbt)
    model.eval()
    return out


T_LOGITS = {a: forward_logits(m) for a, m in TEACHERS.items()}


def audit(model, arch, mode="eval"):
    lg = forward_logits(model) if mode == "eval" else forward_logits_batch_stats(model)
    a = full_model_audit(lg, EX_Y, taxonomy, DEFAULT_COST_PROFILES)
    aw, _ = action_weighted_boundary_inversion_rate(T_LOGITS[arch], lg, EX_Y, robust_graph)
    return {"b2a": float(a["benign_to_attack_rate"]), "a2b": float(a["attack_to_benign_rate"]), "family_f1": float(a["family_macro_f1"]), "awbir": float(aw)}


def bn_var_max(model):
    v = [m.running_var.max().item() for m in model.modules() if isinstance(m, nn.BatchNorm1d)]
    return float(max(v)) if v else float("nan")


def raw_student(arch, method):
    p = (R / f"17b_calibrated_checkpoint_freeze/{method}_r40cal_removed_groups.csv") if arch == "shallow" else (R / f"20b_depth_checkpoint_freeze/{method}_minimal_r40_removed_groups.csv")
    rm = pd.read_csv(p); pm = {str(l): sorted(g["channel_index"].astype(int).tolist()) for l, g in rm.groupby("module_path")}
    st, _ = prune_cnn1d_channels(TEACHERS[arch], pm, EXAMPLE_INPUT, minimum_remaining_per_layer=MIN_W[arch])
    return st.to(DEVICE)


def make_loader(subset_seed, order_seed=None, batch=1024):
    order_seed = subset_seed if order_seed is None else order_seed
    sub = torch.randperm(N_TRAIN, generator=torch.Generator().manual_seed(subset_seed))[: int(N_TRAIN * SUBSET_FRACTION)]
    return torch.utils.data.DataLoader(torch.utils.data.Subset(TRAIN_LOADER.dataset, sub.tolist()), batch_size=batch, shuffle=True,
                                       generator=torch.Generator().manual_seed(order_seed))


def recover(model, arch, loader, n_units, weights=None, tag="", trace=False, has_bn=True):
    """Recover `model` in place; return per-epoch rows (and per-batch trace rows if requested)."""
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    lossf = nn.CrossEntropyLoss(weight=weights) if weights is not None else nn.CrossEntropyLoss()
    rows, trace_rows = [], []
    for unit in range(1, n_units + 1):
        for bi, (xb, yb) in enumerate(loader):
            model.train(); xb = xb.to(DEVICE)
            opt.zero_grad(); lossf(model(xb), yb.to(DEVICE)).backward(); opt.step()
            if trace:
                tr = {"unit": unit, "batch": bi, "batch_max_abs_z": float(xb.abs().max()), "bn_var_max": bn_var_max(model)}
                if bi % 25 == 0:
                    tr["eval_b2a"] = audit(model, arch, "eval")["b2a"]
                trace_rows.append(tr)
        ev = audit(model, arch, "eval")
        bs = audit(model, arch, "batch") if has_bn else None
        col = bool(ev["b2a"] > COLLAPSE_B2A and bs["b2a"] <= COLLAPSE_B2A) if has_bn else bool(ev["b2a"] > COLLAPSE_B2A)
        rows.append({"tag": tag, "architecture": arch, "unit": unit, "eval_b2a": ev["b2a"], "batch_b2a": (bs["b2a"] if has_bn else float("nan")),
                     "eval_family_f1": ev["family_f1"], "eval_awbir": ev["awbir"], "is_collapse": col, "bn_var_max": bn_var_max(model)})
    model.eval()
    return rows, trace_rows


# runtime proofs (probe non-destructive and discriminative; clipping wrapper exposes inner BN modules)
for _a, _m in TEACHERS.items():
    _b = {n: (x.running_mean.clone(), x.running_var.clone()) for n, x in _m.named_modules() if isinstance(x, nn.BatchNorm1d)}
    _e1 = forward_logits(_m); _p = forward_logits_batch_stats(_m); _e2 = forward_logits(_m)
    for n, x in _m.named_modules():
        if isinstance(x, nn.BatchNorm1d): assert torch.equal(x.running_mean, _b[n][0]) and torch.equal(x.running_var, _b[n][1])
    assert np.allclose(_e1, _e2, atol=1e-5) and not _m.training and not np.allclose(_e1, _p, atol=1e-6)
_c = Clipped(copy.deepcopy(TEACHERS["shallow"]), LO, HI).to(DEVICE)
assert sum(isinstance(m, nn.BatchNorm1d) for m in _c.modules()) == 2 and not np.isnan(bn_var_max(_c))
print("helpers verified; clipping wrapper exposes", sum(isinstance(m, nn.BatchNorm1d) for m in _c.modules()), "BatchNorm modules")

JOBS = [(f"pruned/{a}/{m}/s{s}", a, s, ("pruned", m)) for a, m, s in PRUNED_CELLS] + [(f"dense/{a}/s{s}", a, s, ("dense", None)) for a, s in DENSE_CELLS]


In [ ]:
# Stage 4 - baseline reproduction (four pruned, two dense) with the per-batch trace on shallow fisher s307
BASE = OUT / "baseline_epochs.csv"; TRACE = OUT / "baseline_trace_shallow_fisher_s307.csv"
rows = pd.read_csv(BASE).to_dict("records") if BASE.exists() else []
done = {r["tag"] for r in rows}
for tag, arch, seed, (kind, method) in JOBS:
    if tag in done:
        continue
    torch.manual_seed(seed); np.random.seed(seed)
    model = raw_student(arch, method) if kind == "pruned" else copy.deepcopy(TEACHERS[arch]).to(DEVICE)
    want_trace = (tag == "pruned/shallow/fisher/s307") and not TRACE.exists()
    ep, tr = recover(model, arch, make_loader(seed), E_MAX[arch], weights=CLASS_W, tag=tag, trace=want_trace)
    rows.extend(ep); pd.DataFrame(rows).to_csv(BASE, index=False)
    if want_trace:
        pd.DataFrame(tr).to_csv(TRACE, index=False)
    print(f"baseline {tag}: collapse at {[r['unit'] for r in ep if r['is_collapse']] or 'none'}")
base = pd.DataFrame(rows)
n_base = int(base.is_collapse.sum()); print("baseline collapses (six cells):", n_base)
assert n_base > 0, "baseline reproduced no collapse; the ablation would be uninformative"


In [ ]:
# Stage 5 - T1: input clipping at the 0.1 / 99.9 training percentiles, all six cells
CLIP = OUT / "T1_clipping_epochs.csv"
rows = pd.read_csv(CLIP).to_dict("records") if CLIP.exists() else []
done = {r["tag"] for r in rows}
for tag, arch, seed, (kind, method) in JOBS:
    if tag in done:
        continue
    torch.manual_seed(seed); np.random.seed(seed)
    inner = raw_student(arch, method) if kind == "pruned" else copy.deepcopy(TEACHERS[arch]).to(DEVICE)
    model = Clipped(inner, LO, HI).to(DEVICE)
    ep, _ = recover(model, arch, make_loader(seed), E_MAX[arch], weights=CLASS_W, tag=tag)
    rows.extend(ep); pd.DataFrame(rows).to_csv(CLIP, index=False)
    print(f"clipped {tag}: collapse at {[r['unit'] for r in ep if r['is_collapse']] or 'none'} | max BN var {max(r['bn_var_max'] for r in ep):.2f}")
clip = pd.DataFrame(rows); n_base = int(pd.read_csv(OUT / "baseline_epochs.csv").is_collapse.sum())
print("T1 collapses under clipping:", int(clip.is_collapse.sum()), "| baseline:", n_base)


In [ ]:
# Stage 6 - T2: dense GroupNorm models (no running statistics), trained under the NB29 recipe, recovered on the collapsing seeds
GN = OUT / "T2_groupnorm_epochs.csv"
rows = pd.read_csv(GN).to_dict("records") if GN.exists() else []
done = {r["tag"] for r in rows}
GN_MODELS = {}
for arch, seed in DENSE_CELLS:
    ck = MODEL_DIR / f"{arch}_groupnorm_teacher_seed0.pt"
    m = make_cnn("gn")[arch]()
    if ck.exists():
        m.load_state_dict(torch.load(ck, map_location="cpu", weights_only=False)["state_dict"]); m = m.to(DEVICE).eval()
    else:
        torch.manual_seed(0); np.random.seed(0); m = m.to(DEVICE)
        opt = torch.optim.Adam(m.parameters(), lr=1e-3); lossf = nn.CrossEntropyLoss(weight=CLASS_W)
        for _ in range(4):
            m.train()
            for xb, yb in TRAIN_LOADER:
                opt.zero_grad(); lossf(m(xb.to(DEVICE)), yb.to(DEVICE)).backward(); opt.step()
        m.eval(); torch.save({"state_dict": m.cpu().state_dict()}, ck); m = m.to(DEVICE).eval()
    GN_MODELS[arch] = m
    a = audit(m, arch, "eval"); print(f"GroupNorm {arch} teacher: b2a={a['b2a']:.4f} famF1={a['family_f1']:.3f} (BatchNorm teacher's students collapse at these seeds)")
    tag = f"groupnorm_dense/{arch}/s{seed}"
    if tag in done:
        continue
    torch.manual_seed(seed); np.random.seed(seed)
    model = copy.deepcopy(m).to(DEVICE)
    ep, _ = recover(model, arch, make_loader(seed), E_MAX[arch], weights=CLASS_W, tag=tag, has_bn=False)
    for r in ep:
        r["teacher_b2a"] = a["b2a"]                    # the GroupNorm teacher's own baseline, for the relative reading
    rows.extend(ep); pd.DataFrame(rows).to_csv(GN, index=False)
    print(f"  recovery {tag}: eval-mode benign escalation per unit {[round(r['eval_b2a'], 3) for r in ep]} | spikes > 0.10 at {[r['unit'] for r in ep if r['is_collapse']] or 'none'}")
gn = pd.DataFrame(rows); print("T2 spikes:", int(gn.is_collapse.sum()))


In [ ]:
# Stage 7 - T3: subset versus order, shallow fisher
SO = OUT / "T3_subset_vs_order_epochs.csv"
rows = pd.read_csv(SO).to_dict("records") if SO.exists() else []
done = {r["tag"] for r in rows}
variants = [(f"subset_fixed_307/order_{o}", 307, o) for o in (1, 2, 3)] + [(f"order_fixed_307/subset_{s}", s, 307) for s in (11, 12, 13)]
for tag, subset_seed, order_seed in variants:
    if tag in done:
        continue
    torch.manual_seed(subset_seed); np.random.seed(subset_seed)
    model = raw_student("shallow", "fisher")
    ep, _ = recover(model, "shallow", make_loader(subset_seed, order_seed), E_MAX["shallow"], weights=CLASS_W, tag=tag)
    rows.extend(ep); pd.DataFrame(rows).to_csv(SO, index=False)
    print(f"{tag}: collapse at {[r['unit'] for r in ep if r['is_collapse']] or 'none'}")
so = pd.DataFrame(rows)
subset_fixed = int(so[so.tag.str.startswith("subset_fixed")].groupby("tag")["is_collapse"].any().sum())
order_fixed = int(so[so.tag.str.startswith("order_fixed")].groupby("tag")["is_collapse"].any().sum())
print(f"T3: variants collapsing with subset fixed (order varied): {subset_fixed}/3 | with order fixed (subset varied): {order_fixed}/3")


In [ ]:
# Stage 8 - T4: unweighted cross-entropy on the four pruned cells
UW = OUT / "T4_unweighted_epochs.csv"
rows = pd.read_csv(UW).to_dict("records") if UW.exists() else []
done = {r["tag"] for r in rows}
for arch, method, seed in PRUNED_CELLS:
    tag = f"unweighted/{arch}/{method}/s{seed}"
    if tag in done:
        continue
    torch.manual_seed(seed); np.random.seed(seed)
    model = raw_student(arch, method)
    ep, _ = recover(model, arch, make_loader(seed), E_MAX[arch], weights=None, tag=tag)
    rows.extend(ep); pd.DataFrame(rows).to_csv(UW, index=False)
    print(f"{tag}: collapse at {[r['unit'] for r in ep if r['is_collapse']] or 'none'} | final famF1 {ep[-1]['eval_family_f1']:.3f}")
uw = pd.DataFrame(rows); print("T4 collapses without class weighting:", int(uw.is_collapse.sum()))


In [ ]:
# Stage 9 - verdict, including the per-batch trace analysis (T5)
base = pd.read_csv(OUT / "baseline_epochs.csv"); clip = pd.read_csv(OUT / "T1_clipping_epochs.csv")
gn = pd.read_csv(OUT / "T2_groupnorm_epochs.csv"); so = pd.read_csv(OUT / "T3_subset_vs_order_epochs.csv"); uw = pd.read_csv(OUT / "T4_unweighted_epochs.csv")
trace = pd.read_csv(OUT / "baseline_trace_shallow_fisher_s307.csv")
diag = json.load(open(OUT / "extreme_value_diagnostic.json"))

T1 = bool(clip.is_collapse.sum() == 0)
T2 = bool(gn.is_collapse.sum() == 0)
T2_relative_spikes = int(((gn.eval_b2a - gn.teacher_b2a) > 0.10).sum()) if "teacher_b2a" in gn.columns else None
sf = int(so[so.tag.str.startswith("subset_fixed")].groupby("tag")["is_collapse"].any().sum())
of = int(so[so.tag.str.startswith("order_fixed")].groupby("tag")["is_collapse"].any().sum())
T3_reading = "tracks_subset" if (sf >= 2 and of <= 1) else "tracks_order" if (of >= 2 and sf <= 1) else "neither"
T4 = bool(uw.is_collapse.sum() > 0)
# T5: within each unit, rank batches by the jump in max running variance and by max |z|
trace = trace.sort_values(["unit", "batch"]).reset_index(drop=True)
trace["var_jump"] = trace.groupby("unit")["bn_var_max"].diff().fillna(0.0)
top_units = base[(base.tag == "pruned/shallow/fisher/s307") & base.is_collapse]["unit"].tolist()
t5 = {}
for u in sorted(trace.unit.unique()):
    tu = trace[trace.unit == u]
    top_jump = set(tu.nlargest(3, "var_jump")["batch"]); top_z = set(tu.nlargest(3, "batch_max_abs_z")["batch"])
    t5[int(u)] = {"top3_var_jump_batches": sorted(int(b) for b in top_jump), "top3_max_abs_z_batches": sorted(int(b) for b in top_z),
                  "overlap": len(top_jump & top_z), "max_abs_z_in_unit": float(tu.batch_max_abs_z.max()), "max_var_jump": float(tu.var_jump.max())}
T5 = all(t5[u]["overlap"] >= 2 for u in top_units) if top_units else None

verdict = {"arm": "T_trigger_isolation_ciciot2023",
           "baseline_collapses": int(base.is_collapse.sum()), "baseline_cells_collapsing": int(base.groupby("tag")["is_collapse"].any().sum()),
           "T1_clipping_removes_all_collapses": T1, "T1_collapses_under_clipping": int(clip.is_collapse.sum()),
           "T1_max_bn_var_baseline": float(base.bn_var_max.max()), "T1_max_bn_var_clipped": float(clip.bn_var_max.max()),
           "T2_groupnorm_no_spike": T2, "T2_spikes": int(gn.is_collapse.sum()),
           "T2_groupnorm_teacher_b2a": ({t: float(g.teacher_b2a.iloc[0]) for t, g in gn.groupby("tag")} if "teacher_b2a" in gn.columns else None),
           "T2_spikes_relative_to_teacher_baseline": T2_relative_spikes,
           "T2_groupnorm_eval_b2a_by_unit": {t: g.sort_values("unit")["eval_b2a"].round(4).tolist() for t, g in gn.groupby("tag")},
           "T3_reading": T3_reading, "T3_subset_fixed_variants_collapsing": sf, "T3_order_fixed_variants_collapsing": of,
           "T4_collapse_persists_unweighted": T4, "T4_collapses": int(uw.is_collapse.sum()),
           "T5_var_jumps_coincide_with_extreme_batches": T5, "T5_by_unit": t5, "collapse_units_baseline_shallow_fisher_s307": top_units,
           "extreme_value_diagnostic": {k: v for k, v in diag.items() if k not in ("max_abs_z_per_feature", "clip_lo", "clip_hi")},
           "prereg": json.load(open(OUT / "T_PREREGISTRATION.json"))}
(OUT / "T_verdict.json").write_text(json.dumps(verdict, indent=2))
print(json.dumps({k: v for k, v in verdict.items() if k not in ("prereg", "T5_by_unit", "T2_groupnorm_eval_b2a_by_unit")}, indent=2))
for u in top_units: print(f"T5 unit {u}:", t5[u])


In [ ]:
# Stage 10 - figures
trace = pd.read_csv(OUT / "baseline_trace_shallow_fisher_s307.csv").sort_values(["unit", "batch"])
fig, ax1 = plt.subplots(figsize=(10, 3.4)); x = np.arange(len(trace))
ax1.plot(x, trace.bn_var_max, color="#c44e52", lw=1.0, label="max BN running variance"); ax1.set_ylabel("running variance", color="#c44e52")
ax2 = ax1.twinx(); ax2.plot(x, trace.batch_max_abs_z, color="0.3", lw=0.6, alpha=0.7, label="batch max |z|"); ax2.set_ylabel("batch max |z|")
ev = trace.dropna(subset=["eval_b2a"]) if "eval_b2a" in trace.columns else trace.iloc[0:0]
ax3 = ax1.twinx(); ax3.spines["right"].set_position(("axes", 1.08)); ax3.scatter(ev.index.map(lambda i: list(trace.index).index(i)), ev.eval_b2a, color="#4c72b0", s=14, label="eval benign escalation")
ax3.set_ylabel("benign escalation", color="#4c72b0"); ax3.set_ylim(0, 1.05)
for u in trace.unit.unique():
    ax1.axvline(list(trace.index).index(trace[trace.unit == u].index[0]), color="0.8", lw=0.6)
ax1.set_xlabel("batch (units separated by grey lines)"); ax1.set_title("Baseline shallow Fisher s307: per-batch trace")
fig.tight_layout(); fig.savefig(OUT / "T5_trace.png", dpi=200); plt.show()

base = pd.read_csv(OUT / "baseline_epochs.csv"); clip = pd.read_csv(OUT / "T1_clipping_epochs.csv"); uw = pd.read_csv(OUT / "T4_unweighted_epochs.csv")
fig, ax = plt.subplots(figsize=(7.2, 3.2))
for lab, df, col in [("baseline", base, "#c44e52"), ("clipped inputs", clip, "#55a868"), ("unweighted", uw, "#4c72b0")]:
    ax.bar(lab, int(df.is_collapse.sum()), color=col)
ax.set_ylabel("normalisation collapses"); ax.set_title("Collapses by condition (baseline and clipping: six cells; unweighted: four)")
fig.tight_layout(); fig.savefig(OUT / "T1_T4_counts.png", dpi=200); plt.show()

gn = pd.read_csv(OUT / "T2_groupnorm_epochs.csv")
fig, ax = plt.subplots(figsize=(6.4, 3.2))
for t, g in gn.groupby("tag"):
    ax.plot(g.sort_values("unit").unit, g.sort_values("unit").eval_b2a, marker="o", label=t)
ax.axhline(COLLAPSE_B2A, color="0.4", ls=":", lw=0.9); ax.set_yscale("symlog", linthresh=1e-3); ax.set_xlabel("unit"); ax.set_ylabel("eval benign escalation")
ax.set_title("GroupNorm dense models on the collapsing seeds"); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "T2_groupnorm.png", dpi=200); plt.show()
print("figures written ->", OUT)
